### В данном файле:
1) вычитываем MNIST-датасет и сохраняем в original.csv
2) получаем для этого датасета отчет. Наша цель понять, есть ли в датасете n/a, дубликаты, категориальные признаки
3) ориентируясь на результатах base_report, получаем дополнительную отчетную информацию по датасету о 0-columns
4) Используем RandomForestClassifier для получение информации о "неважных" признаках
5) Для уменьшения влияния "проклятия размерности" формируем новый датасет с отброшенными "неважными" признаками. Именно для этого сокращенного датасета будем проводить дальнейшие классификации, оценки и визуализации. Сохраняем сокращенный датасет в reduced.csv

In [ ]:
%pip install --upgrade keras tensorflow pandas jinja2 scikit-learn matplotlib optuna xgboost lightgbm 

In [ ]:
%pip install plotly --no-cache-dir --prefer-binary

In [ ]:
%pip install lightgbm

In [4]:
from private.utils import base_report, load_mnist

df_original = load_mnist()
df_original.to_csv('./data/original.csv', index=False)

base_report(df_original, "label")

rows × cols,n/a,duplicates,min/max,0-columns,types
70000×785,0,0,0/255,65,dtypes: uint8(785)


### Вывод:
1) отсутствие n/a и дубликатов, а также общий тип (unit8) для всех столбцов говорит о том, что предваретельно обрабатывать датасет в этом направлении не нужно
2) 65 0-columns - кандидаты на удаление. Дополнительно проверим, что эти самые 65 столбцов должны попасть список на удаление через feature_importance

In [10]:
# 1. Get the boolean Series
mask = (df_original.drop(columns=['label']) == 0).all(axis=0)

# 2. Filter for True values and get the column names
applied_mask = sorted(mask[mask].index.tolist())
print(f"{len(applied_mask)} : {applied_mask}")

65 : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 52, 53, 54, 55, 56, 57, 82, 83, 84, 85, 111, 112, 140, 168, 476, 560, 644, 671, 672, 673, 699, 700, 701, 727, 728, 729, 730, 754, 755, 756, 757, 758, 759, 780, 781, 782, 783]


In [11]:
# X — признаки (DataFrame), y — целевая переменная
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

X = df_original.drop(columns=['label'])
y = df_original['label']
model = RandomForestClassifier().fit(X, y)

# Создаем DataFrame с результатами
df_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Show only features with 0 importance
zero_importance = df_importance[df_importance['importance'] == 0]
sorted_by_features = sorted(zero_importance['feature'].tolist())
print(f"{len(sorted_by_features)} : {sorted_by_features}")

print(set(applied_mask) - set(sorted_by_features))

115 : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 52, 53, 54, 55, 56, 57, 58, 80, 81, 82, 83, 84, 85, 88, 110, 111, 112, 113, 114, 140, 141, 168, 169, 196, 225, 252, 280, 308, 336, 364, 391, 392, 393, 420, 421, 448, 476, 477, 504, 532, 560, 561, 587, 588, 589, 615, 616, 617, 643, 644, 645, 671, 672, 673, 699, 700, 701, 702, 725, 726, 727, 728, 729, 730, 731, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 779, 780, 781, 782, 783]
set()


### Вывод:
1) все 0-columns из первичного репорта оказались с нулевой важностью в feature_importance (об этом свидетельствует пустой set() - разность множеств)
2) количество "неважных" признаков (115) различается от запуска к запуску. Но полагаю, что поскольку допустимо отбрасывать признаки с 0,05 значимостью, то можем брать любой из наборов 0-значимых кандидатов на отбрасывание

In [12]:
df = df_original.drop(columns=sorted_by_features)

df.to_csv("./data/reduced.csv", index=False)
base_report(df, "label")

rows × cols,n/a,duplicates,min/max,0-columns,types
70000×670,0,0,0/255,0,dtypes: uint8(670)


In [18]:
df.describe()

,12,15,35,36,37,38,39,40,41,42,...,771,772,773,774,775,776,777,778,label,anomaly
count,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,...,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,0.001800,0.000129,0.023071,0.043229,0.062243,0.117929,0.171243,0.193414,0.190086,0.206286,...,0.589571,0.479229,0.333600,0.197414,0.099543,0.046629,0.016614,0.012957,4.452429,0.980000
std,0.440064,0.034017,2.029539,2.951033,3.242019,4.820413,5.677191,6.051967,5.897197,6.243543,...,10.501991,9.448936,7.921717,5.991206,4.256304,2.783732,1.561822,1.553796,2.890195,0.198999
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,1.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,1.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,1.000000
max,116.000000,9.000000,254.000000,255.000000,254.000000,255.000000,255.000000,255.000000,255.000000,255.000000,...,255.000000,255.000000,255.000000,254.000000,254.000000,253.000000,253.000000,254.000000,9.000000,1.000000


####  Оценка аномалий
Чисто формально, чтобы попробывать, получаем 1% кандидатов на аномалии. Собственно в нашем случае распознания цифр, думаю, что нет смысла надеяться на нормальное распределение по признакам, поскольку пиксели в цифрах не случайны.
Для другого датасета мы бы повнимательнее посмотрели на полученные ниже строки и воспользовались бы опытом (будущим) для принятия решений о признаках, алгоритмах (выбирали бы менее чувствительные к аномалиям или проводили бы трансформацию признаков, например)

In [17]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df.columns = df.columns.astype(str)
numeric_cols = df.select_dtypes(include=['uint8']).columns

# Обучение модели поиска аномалий
iso = IsolationForest(contamination=0.01, random_state=42)
preds = iso.fit_predict(df[numeric_cols])

# -1 означает аномалию, 1 — норма
df['anomaly'] = preds
anomalies = df[df['anomaly'] == -1]
display(anomalies)

,12,15,35,36,37,38,39,40,41,42,...,771,772,773,774,775,776,777,778,label,anomaly
121,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-1
294,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,-1
440,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-1
464,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-1
872,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69095,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-1
69123,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,-1
69129,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,-1
69494,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6,-1
